# DiffusionCraft — Conditional UNet & DDPM Training

This notebook covers:
1. `CrossAttention` and `UNetConditional` model definitions
2. `DDPMScheduler` for noise addition and reverse diffusion
3. `TextEncoder` (frozen CLIP) and `ImageCaptionDataset`
4. Forward-pass shape verification
5. Training loop
6. Saving / loading checkpoints
7. Inference: text-to-image generation

The UNet operates entirely in **latent space** (`[B, 4, 16, 16]`).  
A pre-trained (or co-trained) VAE encodes images to latents during training,  
and decodes the final denoised latent to a `128×128` RGB image at inference.

## 1. CrossAttention + UNetConditional

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CrossAttention(nn.Module):
    """
    Multi-head cross-attention.
    Query comes from spatial feature map x  -> shape (B, C, H, W).
    Key/Value come from text context        -> shape (B, seq_len, context_dim).
    Output has the same shape as x.
    """
    def __init__(self, d_model: int, context_dim: int, heads: int = 8):
        super().__init__()
        assert d_model % heads == 0, "d_model must be divisible by heads"
        self.heads    = heads
        self.head_dim = d_model // heads
        self.scale    = self.head_dim ** -0.5

        self.to_q   = nn.Linear(d_model,     d_model,     bias=False)
        self.to_k   = nn.Linear(context_dim, d_model,     bias=False)
        self.to_v   = nn.Linear(context_dim, d_model,     bias=False)
        self.to_out = nn.Linear(d_model,     d_model)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        # Flatten spatial dims: (B, C, H*W) -> (B, H*W, C)
        x_flat = x.view(b, c, h * w).permute(0, 2, 1)

        q = self.to_q(x_flat)   # (B, H*W,      d_model)
        k = self.to_k(context)  # (B, seq_len,   d_model)
        v = self.to_v(context)  # (B, seq_len,   d_model)

        # Split into heads: (B, heads, seq, head_dim)
        def split_heads(t):
            bsz, seq, _ = t.shape
            return t.view(bsz, seq, self.heads, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Scaled dot-product attention
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (B, heads, H*W, seq_len)
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)                                # (B, heads, H*W, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, h * w, c)   # (B, H*W, C)
        out = self.to_out(out)                                     # (B, H*W, C)

        # Restore spatial shape
        return out.permute(0, 2, 1).view(b, c, h, w)

In [ ]:
class UNetConditional(nn.Module):
    """
    Lightweight conditional UNet that predicts the noise added to a latent.

    Input:
        x       : noisy latent  (B, 4, 16, 16)
        t       : timestep      (B,)  — float, values in [0, num_timesteps)
        context : CLIP context  (B, 77, 512)

    Output:
        predicted noise         (B, 4, 16, 16)  — same shape as x

    Architecture
    ------------
    Encoder
        down1 Conv2d 4->128, 3x3   (B,128,16,16)  + timestep bias t1
        attn1 CrossAttention(128, 512)
        down2 Conv2d 128->256, 4x4 stride 2  (B,256,8,8)  + timestep bias t2
        attn2 CrossAttention(256, 512)

    Decoder
        up1   ConvTranspose2d 256->128, 4x4 stride 2  (B,128,16,16)
        skip  add h1 (encoder feature at 16x16)
        out   Conv2d 128->4, 3x3

    Timestep conditioning is injected as a learned bias broadcasted over
    spatial dimensions, so every spatial position sees the same time signal.
    Text conditioning is injected through cross-attention at both resolutions.
    """

    def __init__(self, latent_channels: int = 4, context_dim: int = 512):
        super().__init__()

        # ── Timestep embedding ──────────────────────────────────────────────
        # Map scalar t -> 256-dim vector, then project to channel widths
        self.time_mlp   = nn.Sequential(
            nn.Linear(1, 256),
            nn.SiLU(),
            nn.Linear(256, 256),
        )
        self.time_proj1 = nn.Linear(256, 128)   # for down1 (128 ch)
        self.time_proj2 = nn.Linear(256, 256)   # for down2 (256 ch)

        # ── Encoder ─────────────────────────────────────────────────────────
        self.down1 = nn.Conv2d(latent_channels, 128, kernel_size=3, padding=1)   # 16x16
        self.attn1 = CrossAttention(128, context_dim)

        self.down2 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)     # 16 -> 8
        self.attn2 = CrossAttention(256, context_dim)

        # ── Decoder ─────────────────────────────────────────────────────────
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)  # 8 -> 16
        self.out = nn.Conv2d(128, latent_channels, kernel_size=3, padding=1)

    def forward(
        self,
        x: torch.Tensor,       # (B, 4, 16, 16)
        t: torch.Tensor,       # (B,)
        context: torch.Tensor, # (B, 77, 512)
    ) -> torch.Tensor:
        # ── Timestep embeddings ──────────────────────────────────────────────
        t_emb = self.time_mlp(t.float().unsqueeze(-1))           # (B, 256)
        t1 = self.time_proj1(t_emb).unsqueeze(-1).unsqueeze(-1)  # (B, 128, 1, 1)
        t2 = self.time_proj2(t_emb).unsqueeze(-1).unsqueeze(-1)  # (B, 256, 1, 1)

        # ── Encoder ──────────────────────────────────────────────────────────
        h1 = F.silu(self.down1(x) + t1)   # (B, 128, 16, 16)
        h1 = self.attn1(h1, context)       # cross-attend to text

        h2 = F.silu(self.down2(h1) + t2)  # (B, 256, 8, 8)
        h2 = self.attn2(h2, context)       # cross-attend to text

        # ── Decoder + skip connection ────────────────────────────────────────
        h_up = F.silu(self.up1(h2))        # (B, 128, 16, 16)
        h_up = h_up + h1                   # skip: reuse encoder features

        return self.out(h_up)              # (B, 4, 16, 16)

## 2. DDPM Noise Scheduler

Standard linear-beta DDPM schedule (Ho et al., 2020).

* `add_noise` — used during **training** to corrupt a clean latent at a random timestep  
* `step` — used during **inference** to remove predicted noise one step at a time

In [ ]:
import torch
import math


class DDPMScheduler:
    """
    DDPM scheduler with a linear beta schedule.

    Forward process (training):
        q(x_t | x_0) = N( sqrt(alpha_bar_t)*x_0,  (1-alpha_bar_t)*I )
        => x_t = sqrt(alpha_bar_t)*x_0 + sqrt(1-alpha_bar_t)*eps

    Reverse process (inference):
        p(x_{t-1} | x_t) = N( mu_theta(x_t, t), sigma_t^2 * I )
        mu = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-alpha_bar_t) * eps_theta)
    """

    def __init__(
        self,
        num_timesteps: int = 1000,
        beta_start: float = 1e-4,
        beta_end: float   = 0.02,
    ):
        self.num_timesteps = num_timesteps

        # Linear beta schedule
        betas          = torch.linspace(beta_start, beta_end, num_timesteps)
        alphas         = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)          # alpha_bar_t

        # Pre-compute quantities needed for add_noise and step
        self.register("betas",                      betas)
        self.register("alphas",                     alphas)
        self.register("alphas_cumprod",             alphas_cumprod)
        self.register("sqrt_alphas_cumprod",        alphas_cumprod.sqrt())
        self.register("sqrt_one_minus_alphas_cumprod", (1.0 - alphas_cumprod).sqrt())

    # ------------------------------------------------------------------ utils
    def register(self, name: str, tensor: torch.Tensor):
        """Store a schedule tensor as a plain attribute (no grad)."""
        setattr(self, name, tensor)

    def _gather(self, coef: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Index schedule coefficient by timestep and broadcast to (B,1,1,1)."""
        coef = coef.to(t.device)
        return coef[t.long()].view(-1, 1, 1, 1).float()

    # -------------------------------------------------------------- training
    def add_noise(
        self,
        x_0: torch.Tensor,     # clean latent  (B, C, H, W)
        noise: torch.Tensor,   # sampled noise (B, C, H, W)
        t: torch.Tensor,       # timestep indices (B,)
    ) -> torch.Tensor:
        """Return noisy latent x_t given x_0 and noise at timestep t."""
        sqrt_ab   = self._gather(self.sqrt_alphas_cumprod,        t)  # (B,1,1,1)
        sqrt_1mab = self._gather(self.sqrt_one_minus_alphas_cumprod, t)
        return sqrt_ab * x_0 + sqrt_1mab * noise

    # ------------------------------------------------------------- inference
    def step(
        self,
        noise_pred: torch.Tensor,  # UNet output eps_theta  (B, C, H, W)
        t: int,                    # current scalar timestep
        x_t: torch.Tensor,         # noisy latent at step t  (B, C, H, W)
    ) -> torch.Tensor:
        """
        One reverse-diffusion step: x_t -> x_{t-1}.

        Returns x_{t-1} (with noise added when t > 0).
        """
        device = x_t.device
        beta_t       = self.betas[t].to(device)
        alpha_t      = self.alphas[t].to(device)
        alpha_bar_t  = self.alphas_cumprod[t].to(device)

        # Posterior mean
        coef = beta_t / (1.0 - alpha_bar_t).sqrt()
        mean = (1.0 / alpha_t.sqrt()) * (x_t - coef * noise_pred)

        if t == 0:
            return mean

        # Posterior variance = beta_t (original DDPM uses this for every step)
        sigma = beta_t.sqrt()
        return mean + sigma * torch.randn_like(x_t)

## 3. Text Encoder (frozen CLIP)

In [ ]:
import torch
import torch.nn as nn
from transformers import CLIPModel, CLIPTokenizer


class TextEncoder(nn.Module):
    """
    Wraps the CLIP text transformer and adds a trainable linear projection.

    The CLIP backbone is *frozen*; only `projection` receives gradients
    so that the text conditioning space can adapt to the diffusion task
    without destabilising the pre-trained representations.

    Output: (B, 77, embed_dim)  — one vector per token position.
    """

    def __init__(
        self,
        model_name: str = "openai/clip-vit-base-patch32",
        embed_dim: int  = 512,
    ):
        super().__init__()
        self.tokenizer   = CLIPTokenizer.from_pretrained(model_name)
        clip             = CLIPModel.from_pretrained(model_name)
        self.transformer = clip.text_model          # hidden size = 512
        self.projection  = nn.Linear(512, embed_dim)

        # Freeze the CLIP backbone
        for param in self.transformer.parameters():
            param.requires_grad = False

    def forward(self, text_list: list[str], device: torch.device) -> torch.Tensor:
        tokens = self.tokenizer(
            text_list,
            padding="max_length",
            max_length=77,
            truncation=True,
            return_tensors="pt",
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}

        with torch.no_grad():
            hidden = self.transformer(**tokens).last_hidden_state  # (B, 77, 512)

        return self.projection(hidden)   # (B, 77, embed_dim)

## 4. VAE (for encoding images to latents during training)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResNetBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = F.silu(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return F.silu(x + residual)


class VAE(nn.Module):
    """
    Variational Autoencoder.
    Encoder: 128x128 RGB  ->  (B, 4, 16, 16) latent   (8x spatial compression)
    Decoder: (B, 4, 16, 16) latent  ->  128x128 RGB   (Tanh output, range [-1, 1])
    """

    def __init__(self, in_channels: int = 3, latent_channels: int = 4):
        super().__init__()
        # Encoder: 128 -> 64 -> 32 -> 16
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64,  kernel_size=4, stride=2, padding=1),  nn.ReLU(),
            nn.Conv2d(64,          128, kernel_size=4, stride=2, padding=1),  nn.ReLU(),
            nn.Conv2d(128,         256, kernel_size=4, stride=2, padding=1),
            ResNetBlock(256),
            nn.Conv2d(256, latent_channels * 2, kernel_size=3, padding=1),   # mean + logvar
        )
        # Decoder: 16 -> 32 -> 64 -> 128
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1),
            ResNetBlock(256),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),   # output in [-1, 1] to match dataset normalisation
        )

    def encode(self, x: torch.Tensor):
        moments = self.encoder(x)
        mean, logvar = torch.chunk(moments, 2, dim=1)
        logvar = logvar.clamp(-30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        z = mean + torch.randn_like(std) * std
        return z, mean, logvar

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        z, mean, logvar = self.encode(x)
        return self.decode(z), mean, logvar

## 5. Dataset

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms


class ImageCaptionDataset(Dataset):
    """
    Loads (image, caption) pairs from a directory.

    Expected layout::

        data/laion-art_img/
            1001.jpg
            1001.txt   <- caption for 1001.jpg
            1002.jpg
            1002.txt

    Images are resized to `image_size x image_size` and normalised to [-1, 1]
    to match the VAE decoder's Tanh output range.
    """

    def __init__(self, image_folder: str, image_size: int = 128):
        self.image_folder = image_folder
        self.images = [
            f for f in os.listdir(image_folder)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
            and os.path.exists(
                os.path.join(image_folder, f.rsplit(".", 1)[0] + ".txt")
            )
        ]
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),   # [0,1] -> [-1,1]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx: int):
        img_name = self.images[idx]
        img = Image.open(os.path.join(self.image_folder, img_name)).convert("RGB")
        img = self.transform(img)

        txt_path = os.path.join(
            self.image_folder, img_name.rsplit(".", 1)[0] + ".txt"
        )
        with open(txt_path, "r", encoding="utf-8") as f:
            caption = f.read().strip()

        return img, caption


DATA_DIR   = "src/data/laion-art_img"
IMAGE_SIZE = 128
BATCH_SIZE = 8
MAX_SAMPLES = 5000   # set to None to use the full dataset

dataset = ImageCaptionDataset(DATA_DIR, IMAGE_SIZE)
if MAX_SAMPLES is not None:
    dataset = Subset(dataset, range(min(MAX_SAMPLES, len(dataset))))

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Dataset size : {len(dataset)}")
print(f"Batches/epoch: {len(dataloader)}")

## 6. Shape Verification

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Instantiate all components
vae          = VAE().to(device).eval()
text_encoder = TextEncoder().to(device).eval()
unet         = UNetConditional().to(device)
scheduler    = DDPMScheduler()

# --- one batch forward pass ---
images, captions = next(iter(dataloader))
images = images.to(device)

with torch.no_grad():
    # VAE encode
    latents, mean, logvar = vae.encode(images)
    latents = latents * 0.18215                   # latent scaling (matches inference)

    # Text context
    context = text_encoder(list(captions), device)

# Add noise
B  = latents.shape[0]
t  = torch.randint(0, scheduler.num_timesteps, (B,), device=device)
noise  = torch.randn_like(latents)
noisy  = scheduler.add_noise(latents, noise, t)

# UNet forward
noise_pred = unet(noisy, t.float(), context)

print(f"images        : {images.shape}")
print(f"latents       : {latents.shape}")
print(f"context       : {context.shape}")
print(f"timesteps t   : {t.shape}  values in [{t.min()}, {t.max()}]")
print(f"noisy latent  : {noisy.shape}")
print(f"noise_pred    : {noise_pred.shape}")
print()
print("All shapes look correct:", noise_pred.shape == latents.shape)

## 7. Training

In [ ]:
import os
from tqdm import tqdm

# ── Load a pre-trained VAE checkpoint if available ───────────────────────────
VAE_CHECKPOINT = "checkpoints/vae_celeba.pth"
if os.path.exists(VAE_CHECKPOINT):
    try:
        vae.load_state_dict(torch.load(VAE_CHECKPOINT, map_location=device))
        print(f"Loaded VAE from {VAE_CHECKPOINT}")
    except RuntimeError:
        print("Could not load VAE checkpoint (architecture mismatch) — using random weights.")
else:
    print("No VAE checkpoint found — using random weights (results will be noise).")

vae.eval()
text_encoder.eval()

# ── Optimiser (only UNet parameters are trained) ─────────────────────────────
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

# ── Training loop ────────────────────────────────────────────────────────────
EPOCHS = 10

for epoch in range(EPOCHS):
    unet.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, captions in progress_bar:
        images = images.to(device)

        # 1. Encode images to latents with frozen VAE
        with torch.no_grad():
            latents, _, _ = vae.encode(images)   # (B, 4, 16, 16)
            latents = latents * 0.18215           # latent scaling

        # 2. Encode captions with frozen CLIP
        with torch.no_grad():
            context = text_encoder(list(captions), device)  # (B, 77, 512)

        # 3. Sample random timesteps and add noise
        B = latents.shape[0]
        t = torch.randint(0, scheduler.num_timesteps, (B,), device=device)
        noise = torch.randn_like(latents)
        noisy = scheduler.add_noise(latents, noise, t)

        # 4. Predict the noise
        noise_pred = unet(noisy, t.float(), context)

        # 5. MSE loss between predicted and actual noise
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch {epoch+1}  |  Average loss: {avg_loss:.4f}\n")

## 8. Save Checkpoint

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)
checkpoint_path = "checkpoints/unet_final.pth"
torch.save(unet.state_dict(), checkpoint_path)
print(f"Saved UNet checkpoint to {checkpoint_path}")

## 9. Load Checkpoint and Run Inference

Inference runs the **DDPM reverse loop**: starting from pure Gaussian noise,
the UNet iteratively denoises the latent guided by a text prompt. The final
latent is decoded by the VAE into a `128×128` RGB image.

> **Note**: meaningful images require training on a real captioned dataset  
> (e.g. LAION-Art) for enough epochs. Random or lightly trained weights produce  
> noise but confirm the code path works end-to-end.

In [ ]:
import torch
from PIL import Image as PILImage
import numpy as np

# ── Load checkpoint ──────────────────────────────────────────────────────────
unet_loaded = UNetConditional().to(device)
unet_loaded.load_state_dict(torch.load("checkpoints/unet_final.pth", map_location=device))
unet_loaded.eval()
print("UNet checkpoint loaded successfully")


def generate(
    prompt: str,
    steps: int       = 50,
    cfg_scale: float = 1.0,   # classifier-free guidance scale (>1 strengthens prompt)
) -> PILImage.Image:
    """
    Generate a 128x128 image from a text prompt.

    Parameters
    ----------
    prompt    : text description of the desired image
    steps     : number of denoising steps (more steps = slower but often better)
    cfg_scale : classifier-free guidance scale
                1.0 = no guidance, ~7-9 = strong prompt adherence
    """
    with torch.no_grad():
        # Encode prompt (and empty string for unconditional guidance)
        context        = text_encoder([prompt], device)   # (1, 77, 512)
        uncond_context = text_encoder([""],     device) if cfg_scale > 1.0 else None

        # Start from pure noise in latent space
        latents = torch.randn(1, 4, 16, 16, device=device)

        # Build a sub-sampled timestep schedule (evenly spaced, reversed)
        step_size = max(scheduler.num_timesteps // steps, 1)
        timesteps = list(range(0, scheduler.num_timesteps, step_size))[::-1][:steps]

        for t in timesteps:
            t_tensor = torch.tensor([t], device=device).float()

            noise_pred = unet_loaded(latents, t_tensor, context)

            # Classifier-free guidance: interpolate between unconditional and conditional
            if uncond_context is not None:
                uncond_pred = unet_loaded(latents, t_tensor, uncond_context)
                noise_pred  = uncond_pred + cfg_scale * (noise_pred - uncond_pred)

            latents = scheduler.step(noise_pred, t, latents)

        # Decode latent to image
        # Undo latent scaling (0.18215) and decode with VAE
        image = vae.decode(latents / 0.18215)              # (1, 3, 128, 128), Tanh [-1,1]
        image = (image.clamp(-1.0, 1.0) + 1.0) / 2.0      # rescale to [0, 1]
        image = image.squeeze(0).permute(1, 2, 0).cpu().numpy()  # (128, 128, 3)
        image = (image * 255).astype(np.uint8)

    return PILImage.fromarray(image)


# ── Run inference ────────────────────────────────────────────────────────────
prompt    = "a beautiful mountain landscape at sunset"
generated = generate(prompt, steps=50, cfg_scale=7.5)

os.makedirs("images", exist_ok=True)
generated.save("images/generated_unet.png")
print(f"Saved to images/generated_unet.png")
generated